
# Migration V8.1 → format RAW V9.2 (`DOM_EXTRACTION_V1`)

Ce notebook **ne relance jamais Qwen** et ne touche pas aux PDF.

Objectif :
- lire les JSON historiques produits par `DOM_V8_1_QWEN3_6_VL_27B_FP8_BATCH_DATA_QUALITY` ;
- récupérer uniquement la couche d'extraction `page_records[].raw_data` ;
- produire des JSON compatibles avec la Partie 2 V9/V9.2 ;
- conserver les **99 noms de champs exactement identiques** ;
- ne jamais recopier `normalized_data`, `dossier_row` ni `planning_tl` comme données RAW ;
- tracer explicitement que le dossier a été **migré depuis V8.1**, sans prétendre qu'il a été relu par V9.2.

## Point important de conformité

Le `raw_data` de V8.1 est bien antérieur à la normalisation Python. Il peut toutefois contenir
une valeur corrigée par une **relecture Qwen HD** effectuée par V8.1. Cela reste une extraction VLM,
et non une normalisation Python.

Les anciens JSON restent inchangés. La migration écrit dans un répertoire séparé.


In [ ]:

import hashlib
import json
import shutil
from datetime import datetime
from pathlib import Path

import pandas as pd

# ------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------

SOURCE_V8_JSON_DIR = Path("/mnt/data/domiciliations_out/json_dossiers")

# Staging séparé : ne pas mélanger immédiatement anciens et nouveaux JSON.
MIGRATION_ROOT = Path("/mnt/data/domiciliations_v9/00_migration_v8_1_to_v9_2")
TARGET_JSON_DIR = MIGRATION_ROOT / "json_dossiers"

REPORT_CSV = MIGRATION_ROOT / "migration_report.csv"
MANIFEST_JSON = MIGRATION_ROOT / "migration_manifest.json"

# Répertoire RAW natif V9.2. Publication désactivée par défaut.
V9_RAW_JSON_DIR = Path("/mnt/data/domiciliations_v9/01_extraction_raw/json_dossiers")
PUBLISH_TO_V9_RAW = False

# Test d'abord sur 10 JSON. Mettre None après validation.
MAX_FILES = 10
OVERWRITE_MIGRATED = False

EXPECTED_SOURCE_PIPELINE_PREFIX = "DOM_V8_1"
TARGET_SCHEMA_VERSION = "DOM_EXTRACTION_V1"
TARGET_PIPELINE_VERSION = "DOM_V8_1_MIGRATED_TO_DOM_EXTRACTION_V1"
TARGET_FIELD_SCHEMA_HASH = "da633243929ec467ed246f823571e23de5500442a8510a2a341a78b152aa5a5e"
MIGRATION_VERSION = "MIG_V8_1_TO_V9_2_RAW_V1"
CLASSIFICATION_HARD_MIN_CONFIDENCE = 0.90

MIGRATION_ROOT.mkdir(parents=True, exist_ok=True)
TARGET_JSON_DIR.mkdir(parents=True, exist_ok=True)
V9_RAW_JSON_DIR.mkdir(parents=True, exist_ok=True)

print("Source V8.1 :", SOURCE_V8_JSON_DIR)
print("Staging     :", TARGET_JSON_DIR)
print("Publish V9  :", PUBLISH_TO_V9_RAW)


In [ ]:
FIELD_SCHEMA = {'ENGAGEMENT_DOMICILIATION': ['DOM_NOM_RAISON_SOCIAL_CLIENT', 'DOM_COMPTE_LOCAL', 'DOM_ADRESSE_CLIENT', 'DOM_AGENCE_DOMICILIATAIRE', 'DOM_NUMERO_CONTRAT', 'DOM_DUREE_CONTRAT_MOIS', 'DOM_DATE_DEBUT_CONTRAT', 'DOM_DATE_FIN_CONTRAT', 'DOM_NOM_RAISON_SOCIAL_EMPLOYEUR', 'DOM_ADRESSE_EMPLOYEUR', 'DOM_SALAIRE_NET_MENSUEL', 'DOM_PART_TRANSFERABLE', 'DOM_TAUX_TRANSFERABLE', 'DOM_MONTANT_TOTAL_DOMICILIE', 'DOM_DATE_SIGNATURE'], 'CONTRAT_TRAVAIL': ['CTR_REFERENCE_DOCUMENT', 'CTR_TYPE', 'CTR_EMPLOYEUR', 'CTR_ACTIVITE_EMPLOYEUR', 'CTR_DUREE_MOIS', 'CTR_DATE_DEBUT_CONTRAT', 'CTR_POSTE', 'CTR_NOM_PRENOM_TRAVAILLEUR', 'CTR_PERE_NOM_PRENOM', 'CTR_MERE_NOM_PRENOM', 'CTR_NATIONALITE', 'CTR_DATE_NAISSANCE', 'CTR_LIEU_PAYS_NAISSANCE', 'CTR_ADRESSE_ALGERIE', 'CTR_QUALIFICATION', 'CTR_NUMERO_PERMIS_TRAVAIL', 'CTR_DATE_DELIVRANCE_PERMIS', 'CTR_DATE_DEBUT_VALIDITE_PERMIS', 'CTR_DATE_FIN_VALIDITE_PERMIS', 'CTR_SALAIRE_BRUT', 'CTR_SALAIRE_NET', 'CTR_AFFILIATION_SS', 'CTR_NUMERO_EMPLOYEUR', 'CTR_DATE_SIGNATURE', 'CTR_REFERENCE_DOMICILIATION', 'CTR_SIGNATURE_TRAVAILLEUR_PRESENTE', 'CTR_SIGNATURE_EMPLOYEUR_PRESENTE', 'CTR_CACHET_EMPLOYEUR_PRESENT'], 'CONTRAT_SPECIFIQUE': ['CTS_REFERENCE_DOCUMENT', 'CTS_SAP_ID', 'CTS_EMPLOYEUR', 'CTS_ACTIVITE_EMPLOYEUR', 'CTS_DUREE_MOIS', 'CTS_DATE_DEBUT_CONTRAT', 'CTS_POSTE', 'CTS_NOM_PRENOM_TRAVAILLEUR', 'CTS_PERE_NOM_PRENOM', 'CTS_MERE_NOM_PRENOM', 'CTS_NATIONALITE', 'CTS_DATE_NAISSANCE', 'CTS_LIEU_PAYS_NAISSANCE', 'CTS_ADRESSE_ALGERIE', 'CTS_QUALIFICATION', 'CTS_NUMERO_PERMIS_TRAVAIL', 'CTS_DATE_DELIVRANCE_PERMIS', 'CTS_DATE_DEBUT_VALIDITE_PERMIS', 'CTS_DATE_FIN_VALIDITE_PERMIS', 'CTS_LIGNE_SALAIRE_BRUTE', 'CTS_SALAIRE_NET', 'CTS_SALAIRE_NET_ANCIEN', 'CTS_MENTION_AU_LIEU_DE_PRESENTE', 'CTS_PART_TRANSFERABLE', 'CTS_PART_PAYABLE_DZD', 'CTS_NUMERO_SS_PAYS_ORIGINE', 'CTS_NUMERO_SS_ALGERIE', 'CTS_DATE_DOCUMENT', 'CTS_SIGNATURE_TRAVAILLEUR_PRESENTE', 'CTS_SIGNATURE_EMPLOYEUR_PRESENTE', 'CTS_CACHET_EMPLOYEUR_PRESENT', 'CTS_VISA_INSPECTION_TRAVAIL_PRESENT'], 'TITRE_TRAVAIL': ['TTR_NUMERO_PERMIS', 'TTR_NUMERO_MANUSCRIT', 'TTR_POSTE', 'TTR_DUREE', 'TTR_DATE_DEBUT', 'TTR_DATE_FIN', 'TTR_LIEU_TRAVAIL', 'TTR_EMPLOYEUR', 'TTR_ADRESSE_EMPLOYEUR', 'TTR_FAIT_A', 'TTR_DATE_DELIVRANCE', 'TTR_NOM', 'TTR_PRENOM', 'TTR_DATE_NAISSANCE', 'TTR_LIEU_NAISSANCE', 'TTR_PAYS', 'TTR_NATIONALITE', 'TTR_QUALIFICATION', 'TTR_DATE_ENTREE_ALGERIE', 'TTR_PHOTO_PRESENTE', 'TTR_CACHET_PRESENT'], 'PERMIS_TRAVAIL_COUVERTURE': ['PTR_NUMERO_SERIE', 'PTR_WILAYA', 'PTR_CACHET_DIRECTION_EMPLOI_PRESENT']}


In [ ]:

# ------------------------------------------------------------------
# CONTRÔLE STRICT DU SCHÉMA DES 99 CHAMPS
# ------------------------------------------------------------------

computed_hash = hashlib.sha256(
    json.dumps(FIELD_SCHEMA, sort_keys=True, ensure_ascii=False).encode("utf-8")
).hexdigest()

assert sum(len(v) for v in FIELD_SCHEMA.values()) == 99
assert computed_hash == TARGET_FIELD_SCHEMA_HASH, (computed_hash, TARGET_FIELD_SCHEMA_HASH)

ALL_FIELDS = {f for fields in FIELD_SCHEMA.values() for f in fields}

print("✅ Schéma : 99 champs")
print("✅ Hash   :", computed_hash)
for doc_type, fields in FIELD_SCHEMA.items():
    print(f"   {doc_type:28} : {len(fields):2d}")



## Règles de migration

1. `raw_data` V8.1 est la seule source de données métier reprise.
2. `normalized_data`, `data_quality_corrections`, `dossier_row`, `page_rows` et `planning_tl`
   **ne sont pas injectés** dans le RAW V9.2.
3. Une classification V8.1 avec confiance `< 0.90` est marquée
   `classification_review_required = true`, car V9.2 utilise le seuil réglementaire 0.90.
4. Les valeurs RAW sont comparées avant/après migration : aucune valeur connue ne doit changer.
5. Les métriques historiques sont conservées mais les nombres d'appels V9.2 impossibles à reconstruire
   sont laissés à `null` plutôt que d'être inventés.


In [ ]:

def _is_non_null(v):
    return v not in (None, "")

def _to_float_or_none(v):
    try:
        return float(v)
    except Exception:
        return None

def _unique(seq):
    out = []
    for x in seq:
        if x not in out:
            out.append(x)
    return out

def validate_v8_source(d):
    errors = []
    if not isinstance(d, dict):
        errors.append("JSON_NOT_OBJECT")
        return errors
    if not isinstance(d.get("page_records"), list):
        errors.append("PAGE_RECORDS_MISSING")
    if not d.get("source_file"):
        errors.append("SOURCE_FILE_MISSING")
    pv = str(d.get("pipeline_version") or "")
    if EXPECTED_SOURCE_PIPELINE_PREFIX not in pv:
        errors.append(f"UNEXPECTED_SOURCE_PIPELINE={pv}")
    return errors

def migrate_page_record(rec):
    doc_type = rec.get("doc_type")
    expected = FIELD_SCHEMA.get(doc_type, [])
    old_raw = dict(rec.get("raw_data") or {})

    # Conserver exactement les valeurs RAW connues, sans normalisation.
    raw_new = {k: old_raw[k] for k in old_raw if k in expected}

    # Ne pas perdre d'éventuelles clés inattendues : audit séparé.
    unmapped = {k: v for k, v in old_raw.items() if k not in expected}

    conf = _to_float_or_none(rec.get("classification_confidence"))
    classification_review_required = (
        doc_type not in FIELD_SCHEMA
        or conf is None
        or conf < CLASSIFICATION_HARD_MIN_CONFIDENCE
    )

    flags = list(rec.get("quality_flags") or [])
    flags.append("MIGRATED_FROM_V8_1")
    if classification_review_required:
        flags.append("CLASSIFICATION_REVIEW_REQUIRED_LEGACY_V8_1")
    flags = _unique(flags)

    out = {
        "page_num": rec.get("page_num"),
        "width": rec.get("width"),
        "height": rec.get("height"),
        "white_ratio": rec.get("white_ratio"),

        "doc_type": doc_type,
        "titre_detecte": rec.get("titre_detecte"),
        "bloc_identite_present": bool(rec.get("bloc_identite_present")),
        "classification_requalifiee": bool(rec.get("classification_requalifiee")),
        "classification_retry_fullres": bool(rec.get("classification_retry_fullres")),
        "classification_confidence": conf,
        "classification_raw_text": rec.get("classification_raw_text"),

        # V8.1 ne stockait pas le détail par appel comme V9.2.
        "classification_attempts": [],
        "classification_tokens_in": rec.get("classification_tokens_in"),
        "classification_tokens_out": rec.get("classification_tokens_out"),
        "classification_elapsed_s": rec.get("classification_elapsed_s"),
        "classification_review_required": classification_review_required,

        "raw_data": raw_new,

        "extraction_status": rec.get("extraction_status"),
        "extraction_error": rec.get("extraction_error"),

        # Ne pas inventer des attempts V9.2.
        "extraction_raw_text": None,
        "extraction_attempts": [],
        "extraction_strategies": list(rec.get("extraction_strategies") or []),
        "extraction_taux_remplissage": rec.get("extraction_taux_remplissage"),

        "extraction_call_count": None,
        "retry_call_count": None,
        "extraction_tokens_in": rec.get("extraction_tokens_in"),
        "extraction_tokens_out": rec.get("extraction_tokens_out"),
        "extraction_elapsed_s": rec.get("extraction_elapsed_s"),

        # Relectures Qwen V8.1 : utiles pour audit.
        "field_revisions": list(rec.get("field_revisions") or []),
        "critical_fields_missing": list(rec.get("critical_fields_missing") or []),
        "quality_flags": flags,

        "is_virtual_subdocument": bool(rec.get("is_virtual_subdocument")),
        "virtual_parent_doc_type": rec.get("virtual_parent_doc_type"),
        "frontiere_planche": rec.get("frontiere_planche"),

        "migration_page_audit": {
            "source": "V8.1 page_records[].raw_data",
            "normalized_data_copied": False,
            "legacy_extraction_raw_text_available": bool(rec.get("extraction_raw_text")),
            "legacy_classification_attempts_available": False,
            "legacy_extraction_attempts_available": False,
            "unmapped_raw_fields": unmapped,
        },
    }
    return out

def raw_signature_from_v8(d):
    sig = []
    for rec in d.get("page_records") or []:
        dt = rec.get("doc_type")
        raw = rec.get("raw_data") or {}
        for k, v in raw.items():
            if k in FIELD_SCHEMA.get(dt, []):
                sig.append((rec.get("page_num"), dt, k, v))
    return sig

def raw_signature_from_v9(d):
    sig = []
    for rec in d.get("page_records") or []:
        dt = rec.get("doc_type")
        raw = rec.get("raw_data") or {}
        for k, v in raw.items():
            if k in FIELD_SCHEMA.get(dt, []):
                sig.append((rec.get("page_num"), dt, k, v))
    return sig


In [ ]:

def migrate_dossier(v8, source_json_path):
    source_errors = validate_v8_source(v8)
    if source_errors:
        raise ValueError(" | ".join(source_errors))

    migrated_records = [migrate_page_record(r) for r in v8.get("page_records") or []]

    sig_before = raw_signature_from_v8(v8)

    old_stats = dict(v8.get("stats") or {})
    review_count = sum(bool(r.get("classification_review_required")) for r in migrated_records)
    virtual_count = sum(bool(r.get("is_virtual_subdocument")) for r in migrated_records)
    field_revisions_count = sum(len(r.get("field_revisions") or []) for r in migrated_records)

    out = {
        "schema_version": TARGET_SCHEMA_VERSION,
        "field_schema_hash": TARGET_FIELD_SCHEMA_HASH,
        "field_schema": FIELD_SCHEMA,

        "source_file": v8.get("source_file"),
        "source_sha256": v8.get("source_sha256"),

        # Ne pas prétendre que le fichier a été produit nativement par V9.2.
        "pipeline_version": TARGET_PIPELINE_VERSION,

        "extraction_engine": {
            "model": "Qwen3.6-27B-FP8",
            "flash_attn": False,
            "classification_policy": "LEGACY_V8_1_MIGRATED_NOT_RECLASSIFIED",
            "classification_hard_min_confidence_for_migration_review": CLASSIFICATION_HARD_MIN_CONFIDENCE,
            "source_pipeline_version": v8.get("pipeline_version"),
            "migration_only": True,
            "created_at": datetime.now().isoformat(timespec="seconds"),
        },

        "stats": {
            "pages": old_stats.get("pages"),
            "page_records": len(migrated_records),
            "virtual_subdocuments": virtual_count,
            "classification_review_required": review_count,

            # Impossible à reconstruire fidèlement car V8.1 ne stockait pas
            # les attempts par appel comme V9.2.
            "classification_calls": None,
            "extraction_calls": None,
            "retry_calls": None,
            "qwen_calls_total": None,

            "extraction_passes": old_stats.get("extraction_passes"),
            "field_revisions": field_revisions_count,
            "tokens_in": old_stats.get("tokens_in"),
            "tokens_out": old_stats.get("tokens_out"),
            "tokens_total": old_stats.get("tokens_total"),
            "elapsed_s": old_stats.get("elapsed_s"),
        },

        "page_records": migrated_records,

        "migration": {
            "migration_version": MIGRATION_VERSION,
            "migrated_at": datetime.now().isoformat(timespec="seconds"),
            "source_json_file": source_json_path.name,
            "source_pipeline_version": v8.get("pipeline_version"),
            "target_schema_version": TARGET_SCHEMA_VERSION,
            "target_field_schema_hash": TARGET_FIELD_SCHEMA_HASH,
            "raw_data_policy": "COPY_EXACT_V8_1_RAW_DATA_ONLY",
            "normalized_data_copied": False,
            "dossier_row_copied": False,
            "planning_tl_copied": False,
            "note": (
                "raw_data V8.1 est antérieur à la normalisation Python. "
                "Il peut intégrer des relectures Qwen HD réalisées par V8.1."
            ),
            "legacy_stats": old_stats,
        },
    }

    sig_after = raw_signature_from_v9(out)
    if sig_before != sig_after:
        raise AssertionError("RAW_INTEGRITY_MISMATCH")

    out["migration"]["raw_integrity_check"] = "PASS"
    out["migration"]["raw_values_count"] = len(sig_after)

    return out

def validate_v9_contract(d):
    if d.get("schema_version") != TARGET_SCHEMA_VERSION:
        return False, "BAD_SCHEMA_VERSION"
    if d.get("field_schema_hash") != TARGET_FIELD_SCHEMA_HASH:
        return False, "BAD_FIELD_SCHEMA_HASH"
    if not isinstance(d.get("page_records"), list):
        return False, "PAGE_RECORDS_MISSING"
    return True, "OK"

print("✅ Fonctions de migration prêtes")



## Migration batch

Le test est volontairement limité à 10 fichiers.  
Les JSON V8.1 sources ne sont jamais modifiés.


In [ ]:

source_files = sorted(SOURCE_V8_JSON_DIR.glob("*.json"))
if MAX_FILES is not None:
    source_files = source_files[:int(MAX_FILES)]

print("JSON V8.1 sélectionnés :", len(source_files))

report = []
migrated_paths = []

for i, src in enumerate(source_files, 1):
    try:
        v8 = json.loads(src.read_text(encoding="utf-8"))

        target = TARGET_JSON_DIR / src.name
        if target.exists() and not OVERWRITE_MIGRATED:
            existing = json.loads(target.read_text(encoding="utf-8"))
            ok, msg = validate_v9_contract(existing)
            report.append({
                "JSON_SOURCE": src.name,
                "SOURCE_FILE": v8.get("source_file"),
                "STATUS": "SKIP_EXISTING" if ok else "ERROR_EXISTING_INVALID",
                "MESSAGE": msg,
                "PAGES": (v8.get("stats") or {}).get("pages"),
                "PAGE_RECORDS": len(v8.get("page_records") or []),
                "CLASSIFICATION_REVIEW_REQUIRED": None,
                "RAW_VALUES": len(raw_signature_from_v8(v8)),
            })
            print(f"[{i}/{len(source_files)}] ↪ {src.name} : existe déjà")
            continue

        migrated = migrate_dossier(v8, src)
        ok, msg = validate_v9_contract(migrated)
        if not ok:
            raise ValueError(msg)

        target.write_text(
            json.dumps(migrated, ensure_ascii=False, indent=2, default=str),
            encoding="utf-8"
        )
        migrated_paths.append(target)

        review_count = (migrated.get("stats") or {}).get("classification_review_required")
        raw_count = (migrated.get("migration") or {}).get("raw_values_count")

        report.append({
            "JSON_SOURCE": src.name,
            "SOURCE_FILE": migrated.get("source_file"),
            "STATUS": "MIGRATED",
            "MESSAGE": "OK",
            "PAGES": (migrated.get("stats") or {}).get("pages"),
            "PAGE_RECORDS": len(migrated.get("page_records") or []),
            "CLASSIFICATION_REVIEW_REQUIRED": review_count,
            "RAW_VALUES": raw_count,
        })
        print(
            f"[{i}/{len(source_files)}] ✅ {src.name} | "
            f"RAW={raw_count} | review_classif={review_count}"
        )

    except Exception as exc:
        report.append({
            "JSON_SOURCE": src.name,
            "SOURCE_FILE": None,
            "STATUS": "ERROR",
            "MESSAGE": repr(exc),
            "PAGES": None,
            "PAGE_RECORDS": None,
            "CLASSIFICATION_REVIEW_REQUIRED": None,
            "RAW_VALUES": None,
        })
        print(f"[{i}/{len(source_files)}] ❌ {src.name}: {exc}")

df_report = pd.DataFrame(report)
df_report.to_csv(REPORT_CSV, index=False, encoding="utf-8-sig")

manifest = {
    "migration_version": MIGRATION_VERSION,
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "source_dir": str(SOURCE_V8_JSON_DIR),
    "target_dir": str(TARGET_JSON_DIR),
    "target_schema_version": TARGET_SCHEMA_VERSION,
    "target_field_schema_hash": TARGET_FIELD_SCHEMA_HASH,
    "selected_files": len(source_files),
    "migrated": int((df_report["STATUS"] == "MIGRATED").sum()) if not df_report.empty else 0,
    "errors": int((df_report["STATUS"] == "ERROR").sum()) if not df_report.empty else 0,
    "classification_review_pages": int(
        pd.to_numeric(
            df_report.get("CLASSIFICATION_REVIEW_REQUIRED", pd.Series(dtype=float)),
            errors="coerce"
        ).fillna(0).sum()
    ) if not df_report.empty else 0,
}
MANIFEST_JSON.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\n✅ Rapport :", REPORT_CSV)
print("✅ Manifest:", MANIFEST_JSON)
print("✅ JSON    :", TARGET_JSON_DIR)



## Contrôle d'intégrité après écriture

Cette cellule recharge les fichiers sources V8.1 et les fichiers migrés puis compare,
valeur par valeur, les `raw_data` connus.  
**Aucune différence n'est acceptée.**


In [ ]:

integrity_rows = []

for dst in sorted(TARGET_JSON_DIR.glob("*.json")):
    src = SOURCE_V8_JSON_DIR / dst.name
    if not src.exists():
        integrity_rows.append({
            "JSON": dst.name,
            "STATUS": "SOURCE_NOT_FOUND",
            "RAW_BEFORE": None,
            "RAW_AFTER": None,
        })
        continue

    try:
        v8 = json.loads(src.read_text(encoding="utf-8"))
        v9 = json.loads(dst.read_text(encoding="utf-8"))

        sig8 = raw_signature_from_v8(v8)
        sig9 = raw_signature_from_v9(v9)

        status = "PASS" if sig8 == sig9 else "FAIL"
        integrity_rows.append({
            "JSON": dst.name,
            "STATUS": status,
            "RAW_BEFORE": len(sig8),
            "RAW_AFTER": len(sig9),
        })
    except Exception as exc:
        integrity_rows.append({
            "JSON": dst.name,
            "STATUS": f"ERROR: {exc}",
            "RAW_BEFORE": None,
            "RAW_AFTER": None,
        })

df_integrity = pd.DataFrame(integrity_rows)
print(df_integrity.to_string(index=False))

failures = df_integrity[df_integrity["STATUS"] != "PASS"] if not df_integrity.empty else df_integrity
if len(failures):
    print("\n⚠️ Ne pas publier tant que ces écarts ne sont pas expliqués.")
else:
    print("\n✅ Intégrité RAW : 100 % des fichiers contrôlés sont identiques avant/après migration.")



## Publication optionnelle vers le répertoire RAW V9.2

À utiliser **seulement après validation** du staging.

Règle de priorité :
- si un JSON V9.2 natif existe déjà, il n'est jamais écrasé par un ancien JSON migré ;
- seuls les dossiers historiques absents du répertoire V9.2 sont copiés.


In [ ]:

if PUBLISH_TO_V9_RAW:
    copied = 0
    conflicts = 0

    for src in sorted(TARGET_JSON_DIR.glob("*.json")):
        dst = V9_RAW_JSON_DIR / src.name
        if dst.exists():
            conflicts += 1
            print("⚠️ Conflit, V9.2 natif conservé :", dst.name)
            continue

        shutil.copy2(src, dst)
        copied += 1

    print(f"✅ Publiés : {copied}")
    print(f"⚠️ Conflits non écrasés : {conflicts}")
else:
    print("Publication désactivée. Mettre PUBLISH_TO_V9_RAW = True après validation.")



## Après validation des 10 dossiers

1. Vérifier `migration_report.csv`.
2. Vérifier que le contrôle d'intégrité est `PASS` partout.
3. Examiner les pages `classification_review_required = true`.
4. Passer `MAX_FILES = None`.
5. Relancer la migration sur tous les JSON V8.1.
6. Utiliser ces JSON comme entrée de la Partie 2.
7. Ne publier dans `01_extraction_raw/json_dossiers` qu'après validation.

La Partie 2 pourra alors appliquer **les mêmes règles de normalisation et de Data Quality**
aux anciens dossiers migrés et aux nouveaux dossiers nativement extraits par V9.2.
